In [0]:
import dlt
import pyspark.sql.functions as F

@dlt.table(
    name = "bronze_addresses",    
    comment = "This is raw data",
    table_properties = {'quality': 'bronze'} 
)
def bronze_address():
    return spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.inferSchema", "true") \
        .load("/Volumes/circuit_box/landing/circuit_volume/addresses/") \
            .select( '*', 
                    F.col('_metadata.file_path').alias('file_path'),
                    F.current_timestamp().alias('insertion_time')
                    )


In [0]:
import dlt
import pyspark.sql.functions as F

@dlt.table(
    name="silver_address_clean",
    comment="This is cleaned data",
    table_properties={"quality": "silver"},
)
@dlt.expect_or_fail("valid_customer id", "customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_address", "address_line_1 IS NOT NULL")
@dlt.expect("valid post code", "length(postcode) == 5")
def silver_address_clean():
    return spark.readStream.table("LIVE.bronze_addresses").select(
        "customer_id",
        "address_line_1",
        "city",
        "state",
        "postcode",
        F.col("created_date").cast("date").alias("created_date"),
    )

In [0]:
dlt.create_streaming_table(
    name = 'silver_address',
    comment = 'This is silver address table',
    table_properties = {'quality' : 'silver'}
)

In [0]:
dlt.apply_changes(
    target = 'silver_address',
    source = 'silver_address_clean',
    keys = ['customer_id'],
    sequence_by = 'created_date',
    stored_as_scd_type = 2
)